# 可选实验：简单的 Neural Network

本实验将使用 TensorFlow 构建一个小型 neural network。

<figure style="text-align:center;">
  <img src="./images/C2_W1_CoffeeRoasting.png" alt="咖啡烘焙示例" width="400">
</figure>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from lab_utils_common import dlc
from lab_coffee_utils import load_coffee_data, plt_roast, plt_prob, plt_layer, plt_network, plt_output_unit
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)


## 数据集


In [ ]:
X,Y = load_coffee_data();
print(X.shape, Y.shape)

下面绘制咖啡烘焙数据。两个 feature 分别是温度（摄氏度）和时长（分钟）。本实验参考的 [Coffee Roasting at Home](https://www.merchantsofgreencoffee.com/how-to-roast-green-coffee-in-your-oven/) 将烘焙时长设在 12–15 分钟、温度设在 175–260 摄氏度的范围内。温度升高时，合适的时长应相应缩短。


In [ ]:
plt_roast(X,Y)

### 对数据进行 Normalization

经过 normalization 的数据通常能加快参数拟合；相关的 backpropagation（反向传播）将在下周介绍。这与 Course 1 的方法相同：将各个 feature 调整到相近的尺度。

下面使用 Keras 的 [Normalization layer](https://keras.io/api/layers/preprocessing_layers/numerical/normalization/)，步骤如下：

1. 创建 `Normalization` layer；在本例中，它作为预处理工具单独使用，没有加入后面的模型。
2. 调用 `adapt`，计算训练数据的 mean 和 variance，并保存在 layer 中。
3. 使用该 layer 对数据进行 normalization。

以后使用模型预测新数据时，也必须应用同样的 normalization。


In [ ]:
print(f"Temperature Max, Min pre normalization: {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Duration    Max, Min pre normalization: {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")
norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)  # learns mean, variance
Xn = norm_l(X)
print(f"Temperature Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
print(f"Duration    Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")

通过 tile／复制数据扩大训练集，以减少所需的 training epochs。复制不会增加新的独立样本，但会增加每个 epoch 中的更新次数。


In [ ]:
Xt = np.tile(Xn,(1000,1))
Yt= np.tile(Y,(1000,1))   
print(Xt.shape, Yt.shape)   

## TensorFlow 模型


### 模型结构

<figure style="text-align:center;">
  <img src="./images/C2_W1_RoastingNetwork.PNG" alt="两层咖啡烘焙 neural network" width="200">
</figure>

下面构建课程中的 Coffee Roasting Network。它包含两个 layer，都使用 sigmoid activation。


In [ ]:
tf.random.set_seed(1234)  # applied to achieve consistent results
model = Sequential(
    [
        tf.keras.Input(shape=(2,)),
        Dense(3, activation='sigmoid', name = 'layer1'),
        Dense(1, activation='sigmoid', name = 'layer2')
     ]
)

> **说明 1：** `tf.keras.Input(shape=(2,))` 指定预期输入的 shape，使 TensorFlow 能在此时确定 weight 和 bias 的大小，便于查看模型结构。在本课程使用的构建方式中，也可以省略该语句，让框架在 `model.fit` 接收到输入数据时再确定参数大小。
>
> **说明 2：** 为改善 numerical stability（数值稳定性），训练时通常会让最后一层输出 logits，并在 loss 中统一处理 sigmoid。本实验先显式使用 sigmoid，后续实验会详细介绍更稳定的写法。

`model.summary()` 提供网络结构的摘要。


In [ ]:
model.summary()

摘要中的参数数量，对应各个 weight 和 bias array 的元素总数，如下所示。


In [ ]:
L1_num_params = 2 * 3 + 3   # W1 parameters  + b1 parameters
L2_num_params = 3 * 1 + 1   # W2 parameters  + b2 parameters
print("L1 params = ", L1_num_params, ", L2 params = ", L2_num_params  )

下面检查 TensorFlow 创建的 weight 和 bias。$W$ 的 shape 为“输入 feature 数量 × 该层 unit 数量”，$b$ 的元素数与 unit 数量相同。

- 第一层有 3 个 unit，因此 $W$ 的 shape 为 `(2, 3)`，$b$ 有 3 个元素。
- 第二层有 1 个 unit，因此 $W$ 的 shape 为 `(3, 1)`，$b$ 有 1 个元素。


In [ ]:
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print(f"W1{W1.shape}:\n", W1, f"\nb1{b1.shape}:", b1)
print(f"W2{W2.shape}:\n", W2, f"\nb2{b2.shape}:", b2)

下面的语句将在 Week 2 中详细介绍。目前只需了解：

- `model.compile` 指定 loss function 和 optimizer（优化器）。
- `model.fit` 执行基于 gradient 的优化，使参数拟合训练数据。


In [ ]:
model.compile(
    loss = tf.keras.losses.BinaryCrossentropy(),
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01),
)

model.fit(
    Xt,Yt,            
    epochs=10,
)

### Epochs 与 Batches

上面的 `fit` 将 `epochs` 设为 10，表示训练时完整遍历数据集 10 次。运行过程会显示类似的进度：

```
Epoch 1/10
6250/6250 [==============================] - 6s 910us/step - loss: 0.1782
```

`Epoch 1/10` 表示当前正在执行第 1 个 epoch。为提高效率，训练数据被划分为 batch（批次）。本例使用默认的 batch size 32；扩充后的数据集有 200,000 个样本，共 6250 个 batch。第二行中的 `6250/6250` 表示已处理的 batch 数量。


### 更新后的 Weights

拟合完成后，weight 已被更新：


In [ ]:
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

这些值与调用 `model.fit()` 前打印的值不同。训练后的模型应能判断咖啡烘焙效果的好坏。

为便于后面的讨论，下面先加载一次历史训练得到的参数。这样可以减少 TensorFlow 版本变化及不同训练运行带来的差异，使说明与展示的模型一致。

之后你可以注释掉下面的单元，再运行 Notebook，比较自己训练得到的结果。如果上面的训练 loss 已很低（例如 0.002），通常会得到相近的分类效果。


In [ ]:
# After finishing the lab later, you can re-run all 
# cells except this one to see if your trained model
# gets the same results.

# Set weights from a previous run. 
W1 = np.array([
    [-8.94,  0.29, 12.89],
    [-0.17, -7.34, 10.79]] )
b1 = np.array([-9.87, -9.28,  1.01])
W2 = np.array([
    [-31.38],
    [-27.86],
    [-32.79]])
b2 = np.array([15.54])

# Replace the weights from your trained model with
# the values above.
model.get_layer("layer1").set_weights([W1,b1])
model.get_layer("layer2").set_weights([W2,b2])

In [ ]:
# Check if the weights are successfully replaced
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

### Predictions

<img align="left" src="./images/C2_W1_RoastingDecision.PNG" alt="根据烘焙成功概率进行分类" style="width:380px;padding:10px 20px;">

训练好模型后，就可以进行预测。这里的输出表示烘焙效果良好的概率。若要给出类别决策，需要应用 threshold；本例使用 0.5。


先创建输入数据。模型接受一个或多个样本，每行一个样本；由于有两个 feature，输入 matrix 的 shape 为 `(m, 2)`，其中 $m$ 是样本数。

训练输入经过了 normalization，因此测试数据也必须使用同一个 normalization layer 处理。随后调用 `predict` 方法进行预测。


In [ ]:
X_test = np.array([
    [200,13.9],  # positive example
    [200,17]])   # negative example
X_testn = norm_l(X_test)
predictions = model.predict(X_testn)
print("predictions = \n", predictions)

使用 threshold，将概率转换为类别决策：


In [ ]:
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i] >= 0.5:
        yhat[i] = 1
    else:
        yhat[i] = 0
print(f"decisions = \n{yhat}")

也可以用更简洁的写法完成：


In [ ]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")

## 各 Layer 的作用

下面查看各个 unit 的输出，理解它们在咖啡烘焙判断中的作用。我们将绘制不同输入（时长、温度）下的输出。每个 unit 都使用 logistic function，输出位于 0 与 1 之间，图中的颜色深浅表示输出大小。

> 注意：实验中的编号通常从 0 开始，课程视频中的编号有时从 1 开始。


In [ ]:
plt_layer(X,Y.reshape(-1,),W1,b1,norm_l)

颜色分布显示，各个 unit 分别识别不同的“烘焙效果不佳”区域：unit 0 在温度过低时输出较大，unit 1 在时长过短时输出较大，unit 2 在时长与温度组合不合适时输出较大。

这些功能是网络通过 gradient descent 自行学到的，与人们为同一任务设计的判断规则十分相似。


最后一层的输入来自第一层的 3 个输出，因此更难直接可视化。第一层使用 sigmoid，三个输出都位于 0 与 1 之间，可以在这个范围内枚举组合，用 3D 图展示最后一层的输出。

第一层较大的输出对应“烘焙不佳”区域。下图中，三个输入都较小时，最后一层的输出最大，对应“烘焙良好”区域。


In [ ]:
plt_output_unit(W2,b2)

最后一组图展示整个网络的行为。

- 左图：蓝色深浅表示最后一层的原始概率输出，叠加的“X”和“O”是训练数据。
- 右图：应用 threshold 后的类别预测，“X”和“O”表示网络作出的决策。

下面的绘图可能需要一点时间。


In [ ]:
netf= lambda x : model.predict(norm_l(x))
plt_network(X,Y,netf)

## 恭喜完成实验！

你使用 TensorFlow 构建了一个小型 neural network。它展示了网络如何让多个 unit 分别完成部分判断，再组合起来处理较复杂的决策。
